# **Batch Preprocessing and Validation of Monthly Snow Water Equivalent (SWE) rasters with ArcPy Conditional Logic**
### *Applying learning objectives from GEOG-542 to real-world data, creating one step in a data pipeline to analyze and visualize monthly snow dynamics in the Animas River watershed of southwestern Colorado and northwestern New Mexico.*

Two libraries, ArcPy and OS, are imported for use in this code. During testing of the script, the warning messages did not stand out from the rest of the printed status messages. IPython.display was used to update the warning messages to stand out with red font in HTML. 

Environmental variables are set, including setting the workspace to a local subfolder that contains pre-downloaded GeoTIFF files, downloaded from **ClimateEngine.Org's** web application and sourced from DAYMET's SWE mean variable for each month between October 2014 and December 2024. This data download site is available at: [Climate Engine App](https://app.climateengine.org/climateEngine).

The ability to overwrite outputs in the workspace is enabled and outputs being auto-added to the map is disabled since there are 123 rasters that will be validated and processed.

In [26]:
import arcpy
import os
from IPython.display import display, HTML

In [27]:
arcpy.env.workspace = r"C:\Users\dayoh\Documents\NMSU\GEOG542_Programming\snowpack_project\data_raw\snow_rasters"
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
print("Overwrites enabled; auto-add outputs to map disabled")

Overwrites enabled; auto-add outputs to map disabled


Important variables are defined in this block, including setting the area of interest (AOI) and associated custom projection from a shapefile saved in the project folder. A custom projection was created a previous project for this study area and is most suitable when attempting spatial analysis within the AOI. The spatial reference name is printed for reference.

In [28]:
aoi = r"C:\Users\dayoh\Documents\NMSU\GEOG542_Programming\snowpack_project\data_raw\watersheds\StudyAreaExtent_NAD83_ATM.shp"
projected_output_folder = r"C:\Users\dayoh\Documents\NMSU\GEOG542_Programming\snowpack_project\data_processed\projected_rasters"
clipped_output_folder = r"C:\Users\dayoh\Documents\NMSU\GEOG542_Programming\snowpack_project\data_processed\clipped_rasters"
desc_aoi = arcpy.Describe(aoi)
target_projection = desc_aoi.spatialReference
print(target_projection.name)

Animas_TM_NAD83_2011


A list of the rasters (specifically GeoTIFF files) in the workspace is created and assigned to the variable *rasters* with a print statement confirming the total number of rasters in the subfolder and listing their names for easy reference.

In [29]:
rasters = arcpy.ListRasters("*", "TIF")
#rasters = rasters[:1] #test one raster
print(f"Found {len(rasters)} rasters to process.")
print(rasters)

Found 124 rasters to process.
['animas_swe_201410_mean.swe.tif', 'animas_swe_201411_mean.swe.tif', 'animas_swe_201412_mean.swe.tif', 'animas_swe_201501_mean.swe.tif', 'animas_swe_201502_mean.swe.tif', 'animas_swe_201503_mean.swe.tif', 'animas_swe_201504_mean.swe.tif', 'animas_swe_201505_mean.swe.tif', 'animas_swe_201506_mean.swe.tif', 'animas_swe_201507_mean.swe.tif', 'animas_swe_201508_mean.swe.tif', 'animas_swe_201509_mean.swe.tif', 'animas_swe_201510_mean.swe.tif', 'animas_swe_201511_mean.swe.tif', 'animas_swe_201512_mean.swe.tif', 'animas_swe_201601_mean.swe.tif', 'animas_swe_201602_mean.swe.tif', 'animas_swe_201603_mean.swe.tif', 'animas_swe_201604_mean.swe.tif', 'animas_swe_201605_mean.swe.tif', 'animas_swe_201606_mean.swe.tif', 'animas_swe_201607_mean.swe.tif', 'animas_swe_201608_mean.swe.tif', 'animas_swe_201609_mean.swe.tif', 'animas_swe_201610_mean.swe.tif', 'animas_swe_201611_mean.swe.tif', 'animas_swe_201612_mean.swe.tif', 'animas_swe_201701_mean.swe.tif', 'animas_swe_20170

Next, a user-defined function is created which validates each raster against three different conditions needed for proper analysis of this data. Warnings will be printed with the specific file declared in the print statement, only if there is an issue.

A GeoTIFF file that does not follow the proper naming convention has been added to the directory in order to show the functionality of this validation function.

In [30]:
def validate_raster(raster):
    desc = arcpy.Describe(raster)

    spatial_ref = desc.spatialReference.name
    file_format = desc.format

    # Check 1: Spatial Reference
    if spatial_ref == "Unknown":
        display(HTML(f"<span style='color:red'>WARNING: {raster} has unknown coordinate system.</span>"))
        return False

    # Check 2: Raster format
    if file_format != "TIFF":
        display(HTML(f"<span style='color:red'>WARNING: {raster} is not a tiff file.</span>"))
        return False

    # Check 3: Naming convention
    if "swe" not in raster.lower():
        display(HTML(f"<span style='color:red'>WARNING: {raster} does not follow SWE naming convention.</span>"))
        return False

    return True

The next block uses a For Loop to check each data item within the workspace against the user-defined function, `validate_raster()`, running through three conditionals meant to identify unwanted data in the workspace, print a warning, and skip to the next item. The loop uses Python's `enumerate()` function to track progress and print status messages per item in the directory.

This loop also performs two important geoprocessing steps on the validated rasters, first projecting the file to a custom projection used for the area of interested and then clipping the raster to the area of interest.

Once all of the directory has been checked, a final print statement declares the batch processing complete.

In [31]:
for i, raster in enumerate(rasters, 1):

    print("Checking:", raster)

    # Call user-defined function
    if not validate_raster(raster):
        continue

    print(f"Processing {i} of {len(rasters)}: {raster}")

    base_name = os.path.splitext(raster)[0].replace(".swe", "")
    
    # Project raster to custom Transverse Mercator for AOI
    projected_raster = os.path.join(projected_output_folder, base_name + "_prj.tif")

    arcpy.management.ProjectRaster(
        raster,
        projected_raster,
        target_projection
    )

    # Clip raster to AOI
    clipped_raster = os.path.join(clipped_output_folder, base_name + "_clip.tif")
    
    arcpy.management.Clip(
        projected_raster,
        "#",
        clipped_raster,
        aoi,
        "#",
        "ClippingGeometry",
        "MAINTAIN_EXTENT"
    )
    print("Finished processing:", raster)

print("Batch processing complete.")

Checking: animas_swe_201410_mean.swe.tif
Processing 1 of 124: animas_swe_201410_mean.swe.tif
Finished processing: animas_swe_201410_mean.swe.tif
Checking: animas_swe_201411_mean.swe.tif
Processing 2 of 124: animas_swe_201411_mean.swe.tif
Finished processing: animas_swe_201411_mean.swe.tif
Checking: animas_swe_201412_mean.swe.tif
Processing 3 of 124: animas_swe_201412_mean.swe.tif
Finished processing: animas_swe_201412_mean.swe.tif
Checking: animas_swe_201501_mean.swe.tif
Processing 4 of 124: animas_swe_201501_mean.swe.tif
Finished processing: animas_swe_201501_mean.swe.tif
Checking: animas_swe_201502_mean.swe.tif
Processing 5 of 124: animas_swe_201502_mean.swe.tif
Finished processing: animas_swe_201502_mean.swe.tif
Checking: animas_swe_201503_mean.swe.tif
Processing 6 of 124: animas_swe_201503_mean.swe.tif
Finished processing: animas_swe_201503_mean.swe.tif
Checking: animas_swe_201504_mean.swe.tif
Processing 7 of 124: animas_swe_201504_mean.swe.tif
Finished processing: animas_swe_20150

Batch processing complete.
